# Buổi 1 — Layout & Text Detection (Colab)

<a href="https://colab.research.google.com/github/YOUR_USER/dusa/blob/main/notebooks/colab/01_detection_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Run order:
1. **Runtime → Change runtime type → GPU** (T4 or better).
2. Run the four bootstrap cells below (mount Drive, clone repo, install deps, download SROIE) — ~3-5 min.
3. Then run the rest of the notebook top-to-bottom as in the local version.

> ⚙ Edit `REPO_URL` in the clone cell if your fork lives elsewhere.

## Bootstrap (Colab only)


In [ ]:
# (Optional) mount Drive so checkpoints + figures persist across Colab sessions.
# Skip this cell if you only need a one-off run.
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, subprocess
REPO_URL = "https://github.com/YOUR_USER/dusa.git"  # ← edit if your fork is elsewhere
REPO_DIR = "/content/dusa"

if not os.path.isdir(REPO_DIR):
    print('cloning', REPO_URL)
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    print('pulling latest in', REPO_DIR)
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
%cd $REPO_DIR
!ls


In [ ]:
# Colab ships torch + matplotlib + pandas + opencv; install the rest.
%pip install -q timm shapely pyclipper python-doctr[torch] datasets editdistance
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# Download SROIE (≈30 s; 522 train + 130 test receipts).
import os
if not os.path.isdir('data/sroie/images/test'):
    !python scripts/download_sroie.py
else:
    print('SROIE already present')
!ls data/sroie/images/train | head -3 && echo ... && ls data/sroie/images/test | wc -l


In [ ]:
# Same imports as the local notebook — re-uses src/ now that we cd'd into the repo.
import os, sys
PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)
print('cwd:', PROJECT_ROOT)

from src.data.sroie_loader import list_ids, iter_split
from src.detection import DBNetDetector, MixNetDetector, evaluate_image, aggregate, draw_overlay
print('SROIE test images available:', len(list_ids('test')))


## Theory recap

- **DocLayout-YOLO** — YOLO-based page region segmentation (text / table / figure / title). Not benchmarked here; covered as theory in S1.
- **DB-Net** — differentiable binarization, predicts a probability map + threshold map; fast and the standard baseline.
- **MixNet** — mixed depth-wise convolutions + multi-scale feature fusion; designed for challenging scene text (curved, blurred). The lab fine-tunes it on SROIE; until that checkpoint exists, `MixNetDetector` falls back to DocTR's FAST so the benchmark scaffold is testable.

## Visualize SROIE samples

Sanity check on the input format: SROIE GT is **line-level** (one polygon per receipt line) with a `company / date / address / total / other` label. Below: 6 random test receipts with their GT polygons in green.

In [ ]:
import random
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

FIGURES_DIR = Path('reports/figures/01_detection_benchmark')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

test_ids = list_ids('test')
random.seed(0)
viz_ids = random.sample(test_ids, 6)

from src.data.sroie_loader import load_example
viz_samples = [(rid, *load_example(rid, split='test')) for rid in viz_ids]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('SROIE test samples — green = line-level GT', fontsize=13)
for ax, (rid, img, gt) in zip(axes.ravel(), viz_samples):
    ax.imshow(draw_overlay(img, preds=[], gts=gt))
    ax.set_title(f'{rid}  {img.shape[1]}×{img.shape[0]}  lines={len(gt)}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'sroie_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print('saved →', FIGURES_DIR / 'sroie_samples.png')

labels = Counter(b.label for _, _, gt in viz_samples for b in gt)
print('label distribution across these 6 receipts:', dict(labels))

## Run the benchmark

Curriculum says **50 SROIE test images**. The driver below mirrors `python -m src.detection.run_benchmark --limit 50`.

In [ ]:
LIMIT = 50

dbnet  = DBNetDetector()
mixnet = MixNetDetector()  # falls back to FAST until mixnet_sroie_finetuned.pth exists
detectors = [dbnet, mixnet]


In [ ]:
import time
from src.detection import AggregateStats

results = {}
samples = {}  # keep 10 (image, preds, gt) tuples per detector for visualization

for det in detectors:
    per05, per03, sample_buf = [], [], []
    t0 = time.time()
    for rid, img, gt in iter_split('test', limit=LIMIT):
        preds = det.detect(img)
        per05.append(evaluate_image(preds, gt, iou_thresh=0.5))
        per03.append(evaluate_image(preds, gt, iou_thresh=0.3))
        if len(sample_buf) < 10:
            sample_buf.append((rid, img, preds, gt))
    agg05 = aggregate(per05)
    agg03 = aggregate(per03)
    elapsed = time.time() - t0
    results[det.name] = {
        'n_images': agg05.n_images,
        'elapsed_s': round(elapsed, 1),
        'fps': round(agg05.n_images / elapsed, 2) if elapsed else 0.0,
        'iou_0.5': {'P': agg05.precision, 'R': agg05.recall, 'F1': agg05.f1,
                    'tp': agg05.tp, 'fp': agg05.fp, 'fn': agg05.fn},
        'iou_0.3': {'P': agg03.precision, 'R': agg03.recall, 'F1': agg03.f1,
                    'tp': agg03.tp, 'fp': agg03.fp, 'fn': agg03.fn},
    }
    samples[det.name] = sample_buf
    print(f'{det.name:20s}  IoU=.5 F1={agg05.f1:.3f}  IoU=.3 F1={agg03.f1:.3f}  {results[det.name]["fps"]} fps')


## Comparison table


In [ ]:
import pandas as pd
rows = []
for name, r in results.items():
    for iou in ('0.5', '0.3'):
        m = r[f'iou_{iou}']
        rows.append({'detector': name, 'IoU': iou,
                     'P': round(m['P'], 3), 'R': round(m['R'], 3), 'F1': round(m['F1'], 3),
                     'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'],
                     'FPS': r['fps']})
pd.DataFrame(rows)


## Visualize 10 sample overlays

Green = ground truth (SROIE line-level). Orange = detector prediction (word-level for DB-Net / FAST).


In [ ]:
import matplotlib.pyplot as plt

for name, buf in samples.items():
    fig, axes = plt.subplots(2, 5, figsize=(18, 10))
    fig.suptitle(name, fontsize=14)
    for ax, (rid, img, preds, gt) in zip(axes.ravel(), buf):
        ax.imshow(draw_overlay(img, preds, gt))
        ax.set_title(f'{rid}  gt={len(gt)} pred={len(preds)}', fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    out_path = FIGURES_DIR / f'overlays_{name}.png'
    fig.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print('saved →', out_path)

## Granularity caveat

SROIE GT is **line-level**, DocTR predicts **word-level** boxes → per-pair IoU is structurally low (one GT line contains many word predictions). That depresses both P (many FPs per GT line) and F1@0.5. The IoU=0.3 row is the more honest *relative* metric until either:

1. predicted words are clustered into lines, or
2. MixNet is fine-tuned on SROIE's line-level boxes (the actual Buổi 1 lab).

## Save artifacts

Writes the standalone outputs that the curriculum expects (JSON results + markdown report).


In [ ]:
import json
from pathlib import Path
out = Path('reports/benchmarks/detection_results.json')
out.parent.mkdir(parents=True, exist_ok=True)
with open(out, 'w') as f:
    json.dump({'limit': LIMIT, 'detectors': [{**{'detector': k}, **v} for k, v in results.items()]}, f, indent=2, default=float)
print('wrote', out)


## Fine-tune both detectors on SROIE

Real training run, settings sourced from `configs/detection/mixnet_sroie.yaml`: 736×736 input, AdamW, lr 7e-4 with 1000-step linear warmup, weight-decay 1e-4, gradient-norm clipping 5.0, 60 epochs over the full SROIE train split (522 receipts). Two backbones, same head, same loss → curves are directly comparable:

- **DB-Net side** — `mobilenetv3_large_100` (DocTR DBNet's default backbone), batch 8.
- **MixNet side** — `mixnet_l`, batch 4 (8 GB VRAM doesn't fit batch 8 at 736²).

The detection head is still a minimal 1×1 BCE on the shrunk-polygon prob map — the official DB-Net prob+threshold head isn't vendored yet — but training, supervision and evaluation are now end-to-end on real data, not a smoke test.

**Runtime**: ~13 min for MobileNetV3, ~40 min for MixNet on RTX 4060. Lower `MAX_EPOCHS` in the cell if you want a faster pass.

Artifacts written:
- `checkpoints/detection/{dbnet,mixnet}_sroie_bce.pth` — model state + config per backbone (deliberately *not* `mixnet_sroie_finetuned.pth` so `MixNetDetector` keeps falling back to FAST until the official head is vendored).
- `reports/training/01_detection_benchmark/log.json` — per-step loss + run metadata, keyed by backbone.
- `reports/figures/01_detection_benchmark/training_loss.png` — both loss curves on one axis.

In [ ]:
import json, time
from pathlib import Path
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import timm

from src.detection.dataset_sroie import SROIEDetectionDataset

with open('configs/detection/mixnet_sroie.yaml') as f:
    CFG = yaml.safe_load(f)

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
IMAGE_SIZE = CFG['data']['image_height']
MAX_EPOCHS = CFG['train']['max_epochs']
LR         = CFG['train']['lr']
WD         = CFG['train']['weight_decay']
WARMUP     = CFG['train']['warmup_steps']
GRAD_CLIP  = CFG['train']['grad_clip_norm']

if DEVICE == 'cpu':
    print('⚠ no CUDA — full training is impractical on CPU. Lower MAX_EPOCHS or skip this cell.')

# Per-backbone batch size — mixnet_l at 736×736 needs bs<=4 to stay in 8 GB VRAM.
# Keys match the detector names in the `results` dict from the benchmark cell.
BACKBONES = {
    'dbnet_pretrained': {'backbone': 'mobilenetv3_large_100', 'batch_size': 8, 'short': 'dbnet'},
    'mixnet_finetuned': {'backbone': 'mixnet_l',              'batch_size': 8, 'short': 'mixnet'},
}

CKPT_DIR = Path('checkpoints/detection');                   CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR  = Path('reports/training/01_detection_benchmark'); LOG_DIR.mkdir(parents=True, exist_ok=True)
# FIGURES_DIR is defined in the "Visualize SROIE samples" cell.


class BackboneBCE(nn.Module):
    """Any timm backbone (features_only) + minimal 1-channel BCE seg head."""
    def __init__(self, backbone_name: str) -> None:
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True,
                                          features_only=True, out_indices=(2,))
        c = self.backbone.feature_info.channels()[-1]
        self.head = nn.Sequential(
            nn.Conv2d(c, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1),
        )

    def forward(self, x):
        f = self.backbone(x)[-1]
        logits = self.head(f)
        return F.interpolate(logits, size=x.shape[-2:], mode='bilinear', align_corners=False)


train_ds = SROIEDetectionDataset(split='train', image_size=IMAGE_SIZE, augment=False,
                                  min_text_size=CFG['data']['min_text_size'])
print(f'train receipts: {len(train_ds)}  device={DEVICE}  '
      f'img={IMAGE_SIZE}  epochs={MAX_EPOCHS}  lr={LR}  warmup={WARMUP}')

train_runs = {}  # detector_key → {epoch_losses, step_losses, elapsed_s, config, checkpoint}
for det_key, bcfg in BACKBONES.items():
    backbone_name, batch_size = bcfg['backbone'], bcfg['batch_size']
    model = BackboneBCE(backbone_name).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'\n── {det_key} ({backbone_name}, {n_params:.1f}M params, bs={batch_size}) ──')

    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                         num_workers=CFG['data']['num_workers'] if DEVICE == 'cuda' else 0,
                         pin_memory=(DEVICE == 'cuda'), drop_last=True)
    optim  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    steps_per_epoch = len(loader)
    print(f'  steps/epoch={steps_per_epoch}  total_steps≈{steps_per_epoch * MAX_EPOCHS}')

    step_losses, epoch_losses, t0, step = [], [], time.time(), 0
    model.train()
    for epoch in range(MAX_EPOCHS):
        running = 0.0
        for batch in loader:
            img = batch['image'].to(DEVICE, non_blocking=True)
            tgt = batch['prob_map'].to(DEVICE, non_blocking=True)
            # Linear LR warmup.
            warm = min(1.0, (step + 1) / WARMUP) if WARMUP > 0 else 1.0
            for g in optim.param_groups:
                g['lr'] = LR * warm
            loss = F.binary_cross_entropy_with_logits(model(img), tgt)
            optim.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optim.step()
            step_losses.append(float(loss))
            running += float(loss)
            step += 1
        epoch_loss = running / steps_per_epoch
        epoch_losses.append(epoch_loss)
        if epoch == 0 or (epoch + 1) % 5 == 0 or epoch == MAX_EPOCHS - 1:
            print(f'  epoch {epoch + 1:>2}/{MAX_EPOCHS}  loss={epoch_loss:.4f}  '
                  f'elapsed={time.time() - t0:.0f}s')
    elapsed = time.time() - t0
    print(f'  done in {elapsed:.0f}s  final epoch loss={epoch_losses[-1]:.4f}')

    config = {'backbone': backbone_name, 'image_size': IMAGE_SIZE, 'batch_size': batch_size,
              'max_epochs': MAX_EPOCHS, 'lr': LR, 'warmup_steps': WARMUP,
              'weight_decay': WD, 'grad_clip_norm': GRAD_CLIP}
    ckpt_path = CKPT_DIR / f'{bcfg["short"]}_sroie_bce.pth'
    torch.save({'model_state_dict': model.state_dict(),
                'config': config, 'final_loss': epoch_losses[-1]}, ckpt_path)
    print('  saved →', ckpt_path)

    train_runs[det_key] = {'config': config, 'elapsed_s': round(elapsed, 1),
                            'epoch_losses': epoch_losses, 'step_losses': step_losses,
                            'checkpoint': str(ckpt_path)}

    del model, optim, loader
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

log_path = LOG_DIR / 'log.json'
with open(log_path, 'w') as f:
    json.dump({'device': DEVICE, 'runs': train_runs}, f, indent=2)
print('\nsaved →', log_path)

fig, ax = plt.subplots(figsize=(9, 4.5))
for det_key, run in train_runs.items():
    ax.plot(range(1, len(run['epoch_losses']) + 1), run['epoch_losses'], lw=1.4,
            label=f"{det_key} ({run['config']['backbone']})")
ax.set_xlabel('epoch'); ax.set_ylabel('BCE loss (epoch mean)')
ax.set_title(f'Fine-tune on SROIE ({DEVICE}, {MAX_EPOCHS} epochs)')
ax.legend(); ax.grid(alpha=0.3)
loss_png = FIGURES_DIR / 'training_loss.png'
fig.savefig(loss_png, dpi=120, bbox_inches='tight')
plt.show()
print('saved →', loss_png)

## Bảng so sánh — DB-Net pretrained vs MixNet fine-tuned

Evaluate the fine-tuned MixNet checkpoint (`mixnet_sroie_bce.pth`, produced by the cell above) on the same 50 SROIE test images and put it head-to-head with the DB-Net pretrained baseline (already in `results['dbnet_pretrained']`). The MixNet prob-map output is post-processed to polygons via a DB-Net-style pipeline (threshold → contours → minAreaRect), then scored with the same `evaluate_image` + `aggregate` helpers as the benchmark — so P/R/F1/FPS are apples-to-apples.

Saves `reports/benchmarks/detection_compare.json`.

In [ ]:
import cv2
import numpy as np
import pandas as pd

from src.detection.types import DetectionBox

PROB_THRESH, MIN_AREA = 0.3, 20

def probmap_to_boxes(prob: np.ndarray, target_hw: tuple) -> list:
    """Threshold the prob-map and fit minAreaRect to each connected component."""
    H, W = target_hw
    pm = cv2.resize(prob, (W, H))
    binary = (pm > PROB_THRESH).astype('uint8') * 255
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = []
    for c in contours:
        if cv2.contourArea(c) < MIN_AREA:
            continue
        rect = cv2.minAreaRect(c)
        poly = cv2.boxPoints(rect)
        out.append(DetectionBox(polygon=[(float(p[0]), float(p[1])) for p in poly], score=1.0))
    return out

ckpt = torch.load(CKPT_DIR / 'mixnet_sroie_bce.pth', map_location=DEVICE, weights_only=False)
mixnet_model = BackboneBCE(ckpt['config']['backbone']).to(DEVICE)
mixnet_model.load_state_dict(ckpt['model_state_dict'])
mixnet_model.eval()
img_size = ckpt['config']['image_size']
print('loaded fine-tuned mixnet:', ckpt['config'])

ft_per05, ft_per03 = [], []
t0 = time.time()
for rid, img, gt in iter_split('test', limit=LIMIT):
    H, W = img.shape[:2]
    scale = img_size / max(H, W)
    nh, nw = int(H * scale), int(W * scale)
    canvas = np.zeros((img_size, img_size, 3), dtype=np.uint8)
    canvas[:nh, :nw] = cv2.resize(img, (nw, nh))
    x = torch.from_numpy(canvas).permute(2, 0, 1).float().unsqueeze(0).to(DEVICE) / 255.0
    with torch.no_grad():
        prob = torch.sigmoid(mixnet_model(x))[0, 0, :nh, :nw].cpu().numpy()
    preds = probmap_to_boxes(prob, (H, W))
    ft_per05.append(evaluate_image(preds, gt, iou_thresh=0.5))
    ft_per03.append(evaluate_image(preds, gt, iou_thresh=0.3))
ft_agg05, ft_agg03 = aggregate(ft_per05), aggregate(ft_per03)
ft_fps = ft_agg05.n_images / (time.time() - t0)
print(f'mixnet_finetuned  IoU=.5 F1={ft_agg05.f1:.3f}  IoU=.3 F1={ft_agg03.f1:.3f}  {ft_fps:.2f} fps')

rows = []
for iou in ('0.5', '0.3'):
    m, r = results['dbnet_pretrained'][f'iou_{iou}'], results['dbnet_pretrained']
    rows.append({'detector': 'DB-Net pretrained', 'IoU': iou,
                 'P': round(m['P'], 3), 'R': round(m['R'], 3), 'F1': round(m['F1'], 3),
                 'TP': m['tp'], 'FP': m['fp'], 'FN': m['fn'], 'FPS': r['fps']})
for iou, agg in (('0.5', ft_agg05), ('0.3', ft_agg03)):
    rows.append({'detector': 'MixNet fine-tuned', 'IoU': iou,
                 'P': round(agg.precision, 3), 'R': round(agg.recall, 3), 'F1': round(agg.f1, 3),
                 'TP': agg.tp, 'FP': agg.fp, 'FN': agg.fn, 'FPS': round(ft_fps, 2)})
compare_df = pd.DataFrame(rows)

out = Path('reports/benchmarks/detection_compare.json')
with open(out, 'w') as f:
    json.dump({'limit': LIMIT, 'rows': rows}, f, indent=2, default=float)
print('wrote', out)

compare_df

## Next steps (Buổi 1 lab)

- Vendor the official MixNet architecture under `src/detection/mixnet_arch.py`.
- Build the polygon-bbox training set from `data/sroie/annotations/train/`.
- Train with lr warmup + augmentation; save to `checkpoints/detection/mixnet_sroie_finetuned.pth`.
- Re-run this notebook; `MixNetDetector(checkpoint=…)` will load the real weights and the comparison becomes meaningful (line-level vs line-level).
